In [1]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_add_pool
import pandas as pd
import numpy as np
from rdkit import Chem
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")
print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")

# 1. Atoom Features (Nodes)
def get_atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        int(atom.GetFormalCharge()),
        int(atom.GetIsAromatic()),
        int(atom.GetTotalNumHs())
    ]

# 2. NIEUW: Bindings Features (Edges)
def get_bond_features(bond):
    bond_type = bond.GetBondType()
    return [
        float(bond_type == Chem.rdchem.BondType.SINGLE),
        float(bond_type == Chem.rdchem.BondType.DOUBLE),
        float(bond_type == Chem.rdchem.BondType.TRIPLE),
        float(bond_type == Chem.rdchem.BondType.AROMATIC),
        float(bond.GetIsConjugated()),
        float(bond.IsInRing())
    ]

# 3. Data Conversie
def molecule_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return None
    
    # Koppel de 5 atoom-features
    x = torch.tensor([get_atom_features(atom) for atom in mol.GetAtoms()], dtype=torch.float)
    
    edge_indices = []
    edge_attrs = []
    
    # NIEUW: Lees ook de bindingen uit
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        b_feat = get_bond_features(bond)
        
        # Voeg beide richtingen toe (heen en terug)
        edge_indices.extend([[i, j], [j, i]])
        edge_attrs.extend([b_feat, b_feat])
        
    if len(edge_indices) == 0: return None 
    
    edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attrs, dtype=torch.float) # De 6 extra bindings-kenmerken
    y = torch.tensor([[label]], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

print("\n🧪 Data laden en schalen...")
url = "https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv"
df = pd.read_csv(url)

scaler = StandardScaler()
scaled_labels = scaler.fit_transform(df[['measured log solubility in mols per litre']]).flatten()

data_list = []
for i, row in tqdm(df.iterrows(), total=len(df)):
    graph = molecule_to_graph(row['smiles'], scaled_labels[i])
    if graph is not None:
        data_list.append(graph)

# Split (met vaste seed voor een eerlijke vergelijking met je vorige run)
train_size = int(0.8 * len(data_list))
test_size = len(data_list) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    data_list, [train_size, test_size], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 4. Het "Attention" Model (GATv2)
class GAT_Model(torch.nn.Module):
    def __init__(self):
        super(GAT_Model, self).__init__()
        # GATv2Conv kijkt naar atomen (5 features) EN naar de bindingen (6 features)
        # heads=4 betekent dat het model tegelijkertijd op 4 verschillende manieren naar de data kijkt
        self.conv1 = GATv2Conv(in_channels=5, out_channels=64, heads=4, edge_dim=6)
        
        # 64 channels * 4 heads = 256 input voor de volgende laag
        self.conv2 = GATv2Conv(in_channels=256, out_channels=64, heads=4, edge_dim=6)
        
        # Laatste convolutie, we brengen de heads weer samen (concat=False)
        self.conv3 = GATv2Conv(in_channels=256, out_channels=64, heads=1, concat=False, edge_dim=6)
        
        # Neurale netwerk staartje
        self.lin1 = torch.nn.Linear(64, 32)
        self.lin2 = torch.nn.Linear(32, 1)

    def forward(self, data):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
        
        # We geven nu ook edge_attr mee aan de berekening!
        x = self.conv1(x, edge_index, edge_attr).relu()
        x = self.conv2(x, edge_index, edge_attr).relu()
        x = self.conv3(x, edge_index, edge_attr).relu()
        
        # global_add_pool telt de features van alle atomen op tot één "molecuul-vingerafdruk"
        x = global_add_pool(x, batch)
        
        x = self.lin1(x).relu()
        x = self.lin2(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GAT_Model().to(device)

# Iets agressievere optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=0.002, weight_decay=1e-4)
criterion = torch.nn.MSELoss()

# --- NIEUWE EVALUATIE FUNCTIE ---
def evaluate_model(model, loader, scaler, criterion, device):
    model.eval() # Zet model in test-modus
    total_loss = 0
    y_true = []
    y_pred = []

    with torch.no_grad(): # Geen gradients berekenen (scheelt geheugen en tijd)
        for batch in loader:
            batch = batch.to(device)
            out = model(batch)
            
            # Bereken de loss voor deze test-batch
            loss = criterion(out, batch.y)
            total_loss += loss.item()
            
            # Bewaar de voorspellingen voor de R2 score
            y_true.extend(batch.y.cpu().numpy())
            y_pred.extend(out.cpu().numpy())

    # Reken de schaling terug naar echte scheikundige eenheden
    y_true_real = scaler.inverse_transform(np.array(y_true).reshape(-1, 1))
    y_pred_real = scaler.inverse_transform(np.array(y_pred).reshape(-1, 1))
    
    # Bereken eindscores
    avg_loss = total_loss / len(loader)
    r2 = r2_score(y_true_real, y_pred_real)
    
    return avg_loss, r2


# --- GEÜPGRADEDE TRAINING LOOP ---
epochs = 200
print(f"🔥 Start GAT Training op {device.type.upper()} voor {epochs} Epochs...")

for epoch in range(1, epochs + 1):
    model.train() # Zet model weer in trainings-modus
    train_loss = 0
    
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    
    # Elke 20 epochs printen we NU OOK de testset resultaten!
    if epoch % 20 == 0:
        test_loss, test_r2 = evaluate_model(model, test_loader, scaler, criterion, device)
        print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Test Loss: {test_loss:.4f} | Test R2: {test_r2:.4f}")

# Sla het model op aan het einde!
torch.save(model.state_dict(), "mijn_oplosbaarheid_model.pth")
print("\n💾 Model succesvol opgeslagen als 'mijn_oplosbaarheid_model.pth'")

🚀 GPU: NVIDIA GeForce RTX 5060 Ti

🧪 Data laden en schalen...


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1128/1128 [00:00<00:00, 9372.67it/s]


🔥 Start GAT Training op CUDA voor 200 Epochs...
Epoch  20/200 | Train Loss: 0.3912 | Test Loss: 0.4078 | Test R2: 0.6347
Epoch  40/200 | Train Loss: 0.1247 | Test Loss: 0.1497 | Test R2: 0.8609
Epoch  60/200 | Train Loss: 0.1061 | Test Loss: 0.1285 | Test R2: 0.8801
Epoch  80/200 | Train Loss: 0.0779 | Test Loss: 0.1242 | Test R2: 0.8868
Epoch 100/200 | Train Loss: 0.0754 | Test Loss: 0.1217 | Test R2: 0.8892
Epoch 120/200 | Train Loss: 0.0748 | Test Loss: 0.1014 | Test R2: 0.9063
Epoch 140/200 | Train Loss: 0.0494 | Test Loss: 0.0965 | Test R2: 0.9122
Epoch 160/200 | Train Loss: 0.0620 | Test Loss: 0.1012 | Test R2: 0.9079
Epoch 180/200 | Train Loss: 0.0524 | Test Loss: 0.0991 | Test R2: 0.9087
Epoch 200/200 | Train Loss: 0.0391 | Test Loss: 0.0874 | Test R2: 0.9189

💾 Model succesvol opgeslagen als 'mijn_oplosbaarheid_model.pth'
